# re_nilm — Single-customer appliance detection example

This notebook shows how to use the `re_nilm` library to detect PV, battery, AC, and heat pump for **a single load curve**. No config file required — you work directly with the detector classes.

## What you need
- A 15-min load time series for one customer (DataFrame or CSV/parquet)
- Weather data (temperature + solar radiation) covering the same period

The library provides both: a loader for RE parquet files and two weather backends (MeteoSwiss download or Open-Meteo API).

In [ ]:
# Install the package (only needed once)
# Run this from the repo root, or skip if already installed
# !pip install -e ..

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Check package is importable
import re_nilm
print(f"re_nilm version: {re_nilm.__version__}")

---
## 1. Provide your load curve

### Option A — Use a real RE parquet file

In [ ]:
# Option A: load from a real RE parquet file
# Uncomment and set your path:
#
# from re_nilm.data.loaders.smart_meter import SmartMeterLoader
#
# loader = SmartMeterLoader(
#     data_dir="../data/raw/re",     # directory containing RE parquet files
#     customer_type=None,             # None = load all types, or 'Particuliers'
#     max_annual_kwh=0,               # 0 = no consumption cap
#     n_workers=1,
# )
#
# # Pick one customer from the index
# from re_nilm.pipeline.customer_index import build_customer_file_index, load_customer_from_index
# index = build_customer_file_index("../data/raw/re", cache_path="../data/processed/out/customer_file_index.json")
# customer_id = list(index.keys())[0]   # pick any ID
# meter_df = load_customer_from_index(customer_id, index)
#
# print(f"Loaded customer {customer_id}: {len(meter_df)} rows")
# meter_df.head()

### Option B — Bring your own CSV or parquet

The library expects these columns:

| Column | Type | Description |
|---|---|---|
| `ID` | str | Customer identifier |
| `DT_UTC` | datetime (UTC, tz-naive) | Timestamp |
| `CONSO_KWH` | float | Net consumption per 15-min interval (kWh) |
| `PROD_KWH` | float | PV production injected to grid per 15-min interval (kWh) — 0 if no PV meter |

In [ ]:
# Option B: load from your own file
# Uncomment and adapt:
#
# meter_df = pd.read_csv("your_customer.csv", parse_dates=["DT_UTC"])
# meter_df["ID"] = "my_customer"
#
# Or from parquet:
# meter_df = pd.read_parquet("your_customer.parquet")

### Option C — Synthetic data (demo, no files needed)

In [ ]:
# Synthetic customer: two years at 15-min resolution, with realistic PV production

rng = np.random.default_rng(42)
n = 2 * 365 * 96  # two years

dt = pd.date_range("2022-01-01", periods=n, freq="15min")
hour = dt.hour.to_numpy() + dt.minute.to_numpy() / 60
doy  = dt.dayofyear.to_numpy()

# Smooth solar radiation: peaks at noon in summer, zero at night
solar_angle = np.clip(np.sin(np.pi * (hour - 6) / 12), 0, None)
seasonal    = 0.5 + 0.5 * np.sin(2 * np.pi * (doy - 80) / 365)
rad_W       = np.clip(solar_angle * seasonal * 850, 0, None)

# Realistic consumption: 0.3–0.8 kWh/15min with a slight morning/evening peak
base_load = 0.3 + 0.2 * np.sin(2 * np.pi * (hour - 8) / 24) + rng.normal(0, 0.05, n)

# PV production: proportional to radiation, scaled to ~4 kWp system
pv_capacity_kwp = 4.0
panel_efficiency = 0.15
prod_kw  = (rad_W / 1000.0) * pv_capacity_kwp * panel_efficiency
prod_kwh = (prod_kw * 0.25 + rng.normal(0, 0.005, n)).clip(0)

# Net consumption = total load minus self-consumed PV
conso_kwh = np.clip(base_load - prod_kwh * 0.6, 0, None) + prod_kwh * 0.4

meter_df = pd.DataFrame({
    "ID":       "DEMO_001",
    "DT_UTC":   dt,
    "CONSO_KWH": conso_kwh.astype("float32"),
    "PROD_KWH":  prod_kwh.astype("float32"),
})

print(f"Load curve: {len(meter_df):,} rows  ({meter_df['DT_UTC'].min().date()} → {meter_df['DT_UTC'].max().date()})")
print(f"Total consumption: {meter_df['CONSO_KWH'].sum():.0f} kWh")
print(f"Total production:  {meter_df['PROD_KWH'].sum():.0f} kWh")
meter_df.head(3)

---
## 2. Get weather data

### Option A — Download from Open-Meteo (free, no API key)

In [ ]:
# Option A: fetch from Open-Meteo archive API
# Uncomment and set your coordinates + date range:
#
# from re_nilm.data.loaders.weather import WeatherLoader
#
# loader = WeatherLoader(
#     backend="open_meteo",
#     open_meteo_coords=(46.52, 6.63),   # (lat, lon) — Lausanne
#     cache_dir="../data/processed/out", # saves weather_open_meteo_....parquet
# )
#
# weather_df = loader.load(dt_start="2022-01-01", dt_end="2023-12-31")
# print(f"Weather: {len(weather_df):,} rows")
# weather_df.head(3)

### Option B — Synthetic weather (demo)

In [ ]:
# Synthetic weather matching the synthetic meter data above

temp_C = 10 + 12 * np.sin(2 * np.pi * (doy - 80) / 365) + rng.normal(0, 2, n)

weather_df = pd.DataFrame({
    "dt_utc":       dt,
    "t_2m_C":       temp_C.astype("float32"),
    "global_rad_W": rad_W.astype("float32"),
})

print(f"Weather: {len(weather_df):,} rows")
print(f"Temperature range: {weather_df['t_2m_C'].min():.1f}–{weather_df['t_2m_C'].max():.1f} °C")
print(f"Radiation range: {weather_df['global_rad_W'].min():.0f}–{weather_df['global_rad_W'].max():.0f} W/m²")
weather_df.head(3)

---
## 3. Visualise the raw data

In [ ]:
# Plot one summer week
week = meter_df[(meter_df["DT_UTC"] >= "2022-07-04") & (meter_df["DT_UTC"] < "2022-07-11")]
rad_week = weather_df[(weather_df["dt_utc"] >= "2022-07-04") & (weather_df["dt_utc"] < "2022-07-11")]

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(week["DT_UTC"], week["CONSO_KWH"] * 4, color="steelblue", lw=0.8)  # ×4 → kW
axes[0].set_ylabel("Net consumption (kW)")
axes[0].set_title("Customer DEMO_001 — one summer week")

axes[1].fill_between(week["DT_UTC"], week["PROD_KWH"] * 4, color="orange", alpha=0.7)
axes[1].set_ylabel("PV export (kW)")

axes[2].fill_between(rad_week["dt_utc"], rad_week["global_rad_W"], color="gold", alpha=0.6)
axes[2].set_ylabel("Global radiation (W/m²)")

fig.tight_layout()
plt.show()

---
## 4. PV detection

The PV detector is **unsupervised** — no trained model needed. It computes three indicators:
- `corr_prod_rad`: Pearson correlation between PROD_KWH and solar radiation (high → PV)
- `DeltaProd`: production difference between high-radiation and low-radiation days
- `DeltaNet`: net consumption difference (becomes more negative on sunny days if PV present)

In [ ]:
from re_nilm.detectors.pv import PVDetector

pv_detector = PVDetector(
    corr_threshold=0.3,          # minimum prod-radiation correlation
    delta_net_threshold=-0.1,    # DeltaNet below this → PV flag
    min_yearly_prod_kwh=1.0,     # minimum annual production to consider
)

pv_result = pv_detector.predict_customer(meter_df, weather_df)

print("=== PV Detection Result ===")
for k, v in pv_result.items():
    print(f"  {k:25s}: {v}")

---
## 5. PV capacity estimation

Only runs on PV-positive customers. Uses a bootstrap regression to estimate:
- `pv_capacity_kwp`: installed capacity in kilowatt-peak
- `pv_ci_lower / upper`: 90% bootstrap confidence interval
- `sc_share`: self-consumption fraction (what fraction of PV is used on-site)

In [ ]:
from re_nilm.estimators.pv_capacity import PVCapacityEstimator

cap_estimator = PVCapacityEstimator(
    bootstrap_n=200,       # number of bootstrap iterations (more = tighter CI)
    capacity_min_kwp=0.1,  # estimates below this are reported as NaN
)

cap_result = cap_estimator.estimate(meter_df, pv_result, weather_df)

if cap_result:
    print("=== PV Capacity Estimate ===")
    for k, v in cap_result.items():
        print(f"  {k:25s}: {v:.3f}" if isinstance(v, float) else f"  {k:25s}: {v}")
    print(f"\n  True capacity (synthetic): {pv_capacity_kwp} kWp")
    print(f"  Estimated:                 {cap_result['pv_capacity_kwp']:.2f} kWp")
else:
    print("No PV capacity estimated (customer is PV-negative or insufficient data)")

---
## 6. Battery detection

The battery detector compares load profiles on **dark days** (no solar) vs **sunny days** — a battery shifts the evening peak and reduces midday export. It requires PV to be present (configurable).

In [ ]:
from re_nilm.detectors.battery import BatteryDetector

battery_detector = BatteryDetector(
    classification_threshold=0.5,
    enforce_pv_required=True,      # skip if no PV detected
    dark_day_rad_max_w=100.0,      # days below this are "reference" days
    sunny_day_rad_min_w=100.0,     # days above this are "sunny" days
)

# Pass the PV capacity result as context so the detector knows the PV size
pv_context = cap_result if cap_result else pv_result
batt_result = battery_detector.predict_customer(meter_df, weather_df, pv_result=pv_context)

print("=== Battery Detection Result ===")
for k, v in batt_result.items():
    print(f"  {k:25s}: {v}")

---
## 7. AC and HP detection

These require **trained model files** (`models/ac_detector_v1.joblib`, `models/hp_detector_v1.joblib`). Run `scripts/train_models.py` to produce them.

If the models exist, the pattern is identical to the detectors above:

In [ ]:
from pathlib import Path

# AC detection
ac_model_path = Path("../models/ac_detector_v1.joblib")

if ac_model_path.exists():
    from re_nilm.detectors.ac import ACDetector
    ac_detector = ACDetector.load(
        ac_model_path,
        prob_threshold=0.55,
        inference_months=[6, 7, 8],  # Swiss summer
    )
    ac_result = ac_detector.predict_customer(meter_df, weather_df)
    print("AC result:", ac_result)
else:
    print(f"AC model not found at {ac_model_path}")
    print("Run: python scripts/train_models.py")
    ac_result = {"customer_id": "DEMO_001", "has_ac": False, "prob_ac": 0.0}

# HP detection
hp_model_path = Path("../models/hp_detector_v1.joblib")

if hp_model_path.exists():
    from re_nilm.detectors.heat_pump import HeatPumpDetector
    hp_detector = HeatPumpDetector.load(
        hp_model_path,
        prob_threshold=0.5,
        night_rad_threshold=20.0,  # features computed from nighttime rows only
    )
    hp_result = hp_detector.predict_customer(meter_df, weather_df)
    print("HP result:", hp_result)
else:
    print(f"HP model not found at {hp_model_path}")
    hp_result = {"customer_id": "DEMO_001", "has_hp": False, "prob_hp": 0.0, "hp_type": "no_hp"}

---
## 8. Summary

In [ ]:
summary = {
    "customer_id":         pv_result["customer_id"],
    "has_pv":              pv_result["has_pv"],
    "pv_capacity_kwp":     cap_result["pv_capacity_kwp"]     if cap_result else None,
    "pv_capacity_ci":      f"{cap_result['pv_ci_lower']:.2f}–{cap_result['pv_ci_upper']:.2f} kWp" if cap_result else None,
    "sc_share":            f"{cap_result['sc_share']:.1%}"   if cap_result else None,
    "has_battery":         batt_result["has_battery"],
    "battery_prob":        batt_result["prob_battery"],
    "has_ac":              ac_result["has_ac"],
    "prob_ac":             ac_result["prob_ac"],
    "has_hp":              hp_result["has_hp"],
    "hp_type":             hp_result["hp_type"],
}

print("\n" + "="*45)
print("  Customer appliance detection summary")
print("="*45)
for k, v in summary.items():
    print(f"  {k:25s}: {v}")

---
## 9. Running many customers at once

For a portfolio of thousands of customers, use the `StreamingEngine` directly or the full `PipelineOrchestrator`. This handles batching, checkpointing, and parallel execution automatically.

In [ ]:
# Minimal example: StreamingEngine + PVDetector on a list of customers
#
# from pathlib import Path
# from re_nilm.pipeline.streaming import StreamingEngine
# from re_nilm.pipeline.customer_index import build_customer_file_index, load_customer_from_index
# from re_nilm.detectors.pv import PVDetector
#
# index = build_customer_file_index(
#     data_dir="../data/raw/re",
#     cache_path="../data/processed/out/customer_file_index.json",
# )
#
# detector = PVDetector()
# weather = ...  # load once, shared across all customers
#
# def process_one_customer(customer_id: str):
#     df = load_customer_from_index(customer_id, index)
#     return detector.predict_customer(df, weather)
#
# engine = StreamingEngine(
#     n_workers=4,          # parallel processes
#     batch_size=500,       # customers per autosave batch
#     resume=True,          # skip already-processed customers on restart
#     checkpoint_path=Path("../data/processed/out/pv_indicators_ckpt.parquet"),
# )
#
# results = engine.run(
#     customer_ids=list(index.keys()),
#     processor_fn=process_one_customer,
#     output_path=Path("../data/processed/out/pv_indicators.parquet"),
# )
#
# print(results.head())

print("StreamingEngine example above — uncomment and adapt for your data directory.")
print("Or use the CLI: python scripts/run_pipeline.py --config config/re_production.yaml --detectors pv")